# Amazon Bestselling Books — Analyze Phase

Compute the findings that answer the business question: what characterizes a repeat Amazon
bestseller? Source: `data/processed/bestsellers_clean.parquet` (550 rows, 8 columns, including
`times_on_list`). Small aggregate tables are saved to `data/summary/` for reuse in Share.

## Setup

In [1]:
import pandas as pd
import numpy as np
import os

PROC_DIR = "../data/processed"
SUMMARY_DIR = "../data/summary"
os.makedirs(SUMMARY_DIR, exist_ok=True)

df = pd.read_parquet(os.path.join(PROC_DIR, "bestsellers_clean.parquet"))
print(f"Loaded processed data: {df.shape}")

Loaded processed data: (550, 8)


## Build a title-level table

Row-level data has one row per (title × year). For questions about what characterizes a *title*
(genre, repeat status), row-level aggregation would overweight repeat titles. Build a one-row-per-
title table first, checking that `genre` is consistent for every title (it should never change
across a title's own appearances).

In [2]:
genre_check = df.groupby("name")["genre"].nunique()
print(f"Titles with inconsistent genre across appearances: {(genre_check > 1).sum()}")

titles = df.groupby("name").agg(
    author=("author", "first"),
    genre=("genre", "first"),
    times_on_list=("times_on_list", "first"),
    mean_price=("price", "mean"),
    mean_rating=("user_rating", "mean"),
    mean_reviews=("reviews", "mean"),
).reset_index()
print(f"Title-level table: {titles.shape}")

Titles with inconsistent genre across appearances: 0
Title-level table: (350, 7)


## 1. Distribution of `times_on_list`

How many books are one-hit-wonders vs. repeat bestsellers, and how far does the top repeater
span?

In [3]:
dist = titles["times_on_list"].value_counts().sort_index()
print("times_on_list distribution (titles):")
print(dist)
one_hit = (titles["times_on_list"] == 1).sum()
repeaters = (titles["times_on_list"] > 1).sum()
print(f"\nOne-hit-wonders: {one_hit} ({one_hit/len(titles)*100:.1f}%)")
print(f"Repeat bestsellers (>1 year): {repeaters} ({repeaters/len(titles)*100:.1f}%)")
top_repeaters = titles.sort_values("times_on_list", ascending=False).head(5)
print("\nTop 5 repeat titles:")
print(top_repeaters[["name", "author", "genre", "times_on_list"]].to_string(index=False))
dist.rename("n_titles").to_csv(os.path.join(SUMMARY_DIR, "times_on_list_distribution.csv"))

times_on_list distribution (titles):
times_on_list
1     255
2      55
3      17
4       7
5       8
6       2
7       2
8       1
9       1
10      2
Name: count, dtype: int64

One-hit-wonders: 255 (72.9%)
Repeat bestsellers (>1 year): 95 (27.1%)

Top 5 repeat titles:
                                                                     name                             author       genre  times_on_list
                      The 5 Love Languages: The Secret to Love That Lasts                       Gary Chapman Non Fiction             10
Publication Manual of the American Psychological Association, 6th Edition American Psychological Association Non Fiction             10
                                                      StrengthsFinder 2.0                             Gallup Non Fiction              9
                                                Oh, the Places You'll Go!                          Dr. Seuss     Fiction              8
                                              The 

## 2. Genre split and repeat skew

In [4]:
genre_counts_rows = df["genre"].value_counts()
print(f"Genre counts (rows): {genre_counts_rows.to_dict()}")
genre_counts_titles = titles["genre"].value_counts()
print(f"Genre counts (distinct titles): {genre_counts_titles.to_dict()}")

genre_repeat = titles.groupby("genre").agg(
    n_titles=("name", "count"),
    n_repeaters=("times_on_list", lambda s: (s > 1).sum()),
    mean_times_on_list=("times_on_list", "mean"),
)
genre_repeat["pct_repeaters"] = (genre_repeat["n_repeaters"] / genre_repeat["n_titles"] * 100).round(1)
print("\nGenre repeat-rate breakdown (title-level):")
print(genre_repeat)
genre_repeat.to_csv(os.path.join(SUMMARY_DIR, "genre_repeat_rate.csv"))

Genre counts (rows): {'Non Fiction': 310, 'Fiction': 240}
Genre counts (distinct titles): {'Non Fiction': 190, 'Fiction': 160}

Genre repeat-rate breakdown (title-level):
             n_titles  n_repeaters  mean_times_on_list  pct_repeaters
genre                                                                
Fiction           160           44            1.487500           27.5
Non Fiction       190           51            1.626316           26.8


## 3. Price distribution and correlation with rating / repeat status

In [5]:
print("Price describe (row-level):")
print(df["price"].describe())
corr_price_rating = df["price"].corr(df["user_rating"])
print(f"\nCorrelation price vs user_rating (row-level): r={corr_price_rating:.3f}")
corr_price_tol = titles["mean_price"].corr(titles["times_on_list"])
print(f"Correlation mean_price vs times_on_list (title-level): r={corr_price_tol:.3f}")

price_by_repeat = titles.assign(is_repeat=titles["times_on_list"] > 1).groupby("is_repeat")["mean_price"].agg(["mean", "median", "count"])
print("\nPrice by repeat status (title-level):")
print(price_by_repeat)
price_by_repeat.to_csv(os.path.join(SUMMARY_DIR, "price_by_repeat_status.csv"))

Price describe (row-level):
count    550.000000
mean      13.100000
std       10.842262
min        0.000000
25%        7.000000
50%       11.000000
75%       16.000000
max      105.000000
Name: price, dtype: float64

Correlation price vs user_rating (row-level): r=-0.133
Correlation mean_price vs times_on_list (title-level): r=0.012

Price by repeat status (title-level):
                mean  median  count
is_repeat                          
False      13.282353    12.0    255
True       12.280226    10.0     95


## 4. Rating distribution and repeat status

In [6]:
print("User rating describe (row-level):")
print(df["user_rating"].describe())
corr_rating_tol = titles["mean_rating"].corr(titles["times_on_list"])
print(f"\nCorrelation mean_rating vs times_on_list (title-level): r={corr_rating_tol:.3f}")

rating_by_repeat = titles.assign(is_repeat=titles["times_on_list"] > 1).groupby("is_repeat")["mean_rating"].agg(["mean", "median", "count"])
print("\nRating by repeat status (title-level):")
print(rating_by_repeat)
rating_by_repeat.to_csv(os.path.join(SUMMARY_DIR, "rating_by_repeat_status.csv"))

User rating describe (row-level):
count    550.000000
mean       4.618364
std        0.226980
min        3.300000
25%        4.500000
50%        4.700000
75%        4.800000
max        4.900000
Name: user_rating, dtype: float64

Correlation mean_rating vs times_on_list (title-level): r=0.047

Rating by repeat status (title-level):
               mean  median  count
is_repeat                         
False      4.603922     4.6    255
True       4.620977     4.7     95


## 5. Reviews distribution and correlation — the strongest repeat-bestseller signal

`Reviews` is a proxy for reader engagement/popularity (flagged in Prepare, not necessarily
quality). Check whether it associates with repeat-bestseller status more strongly than price or
rating do.

In [7]:
print("Reviews describe (row-level):")
print(df["reviews"].describe())
corr_reviews_tol = titles["mean_reviews"].corr(titles["times_on_list"])
print(f"\nCorrelation mean_reviews vs times_on_list (title-level): r={corr_reviews_tol:.3f}")
corr_reviews_rating = df["reviews"].corr(df["user_rating"])
corr_reviews_price = df["reviews"].corr(df["price"])
print(f"Correlation reviews vs user_rating (row-level): r={corr_reviews_rating:.3f}")
print(f"Correlation reviews vs price (row-level): r={corr_reviews_price:.3f}")

reviews_by_repeat = titles.assign(is_repeat=titles["times_on_list"] > 1).groupby("is_repeat")["mean_reviews"].agg(["mean", "median", "count"]).round(1)
print("\nReviews by repeat status (title-level):")
print(reviews_by_repeat)
reviews_by_repeat.to_csv(os.path.join(SUMMARY_DIR, "reviews_by_repeat_status.csv"))

Reviews describe (row-level):
count      550.000000
mean     11953.281818
std      11731.132017
min         37.000000
25%       4058.000000
50%       8580.000000
75%      17253.250000
max      87841.000000
Name: reviews, dtype: float64

Correlation mean_reviews vs times_on_list (title-level): r=0.228
Correlation reviews vs user_rating (row-level): r=-0.002
Correlation reviews vs price (row-level): r=-0.109

Reviews by repeat status (title-level):
              mean   median  count
is_repeat                         
False       7321.5   5235.0    255
True       16381.3  11616.0     95


## 6. Top authors by total appearances

In [8]:
author_stats = df.groupby("author").agg(
    total_appearances=("name", "count"),
    distinct_titles=("name", "nunique"),
).sort_values("total_appearances", ascending=False)
print("Top 10 authors by total appearances (rows):")
print(author_stats.head(10))
author_stats.head(15).to_csv(os.path.join(SUMMARY_DIR, "top_authors.csv"))

Top 10 authors by total appearances (rows):
                                    total_appearances  distinct_titles
author                                                                
Jeff Kinney                                        12               12
Gary Chapman                                       11                2
Rick Riordan                                       11               10
Suzanne Collins                                    11                5
American Psychological Association                 10                1
Dr. Seuss                                           9                2
Gallup                                              9                1
Rob Elliott                                         8                2
Dav Pilkey                                          7                6
Bill O'Reilly                                       7                6


## 7. Year-over-year trend in average price / rating / reviews (2009-2019)

In [9]:
year_trend = df.groupby("year").agg(
    avg_price=("price", "mean"),
    avg_rating=("user_rating", "mean"),
    avg_reviews=("reviews", "mean"),
    n_books=("name", "count"),
).round(2)
print("Year-over-year trend (2009-2019):")
print(year_trend)
year_trend.to_csv(os.path.join(SUMMARY_DIR, "year_trend.csv"))

price_change = year_trend["avg_price"].iloc[-1] - year_trend["avg_price"].iloc[0]
rating_change = year_trend["avg_rating"].iloc[-1] - year_trend["avg_rating"].iloc[0]
reviews_change_pct = (year_trend["avg_reviews"].iloc[-1] / year_trend["avg_reviews"].iloc[0] - 1) * 100
print(f"\nAvg price change 2009->2019: {price_change:+.2f} (from {year_trend['avg_price'].iloc[0]:.2f} to {year_trend['avg_price'].iloc[-1]:.2f})")
print(f"Avg rating change 2009->2019: {rating_change:+.2f} (from {year_trend['avg_rating'].iloc[0]:.2f} to {year_trend['avg_rating'].iloc[-1]:.2f})")
print(f"Avg reviews change 2009->2019: {reviews_change_pct:+.1f}% (from {year_trend['avg_reviews'].iloc[0]:.0f} to {year_trend['avg_reviews'].iloc[-1]:.0f})")

Year-over-year trend (2009-2019):
      avg_price  avg_rating  avg_reviews  n_books
year                                             
2009      15.40        4.58      4710.12       50
2010      13.48        4.56      5479.62       50
2011      15.10        4.56      8100.82       50
2012      15.30        4.53     13090.92       50
2013      14.60        4.55     13098.14       50
2014      14.64        4.62     15859.94       50
2015      10.42        4.65     14233.38       50
2016      13.18        4.68     14196.00       50
2017      11.38        4.66     12888.40       50
2018      10.52        4.67     13930.42       50
2019      10.08        4.74     15898.34       50

Avg price change 2009->2019: -5.32 (from 15.40 to 10.08)
Avg rating change 2009->2019: +0.16 (from 4.58 to 4.74)
Avg reviews change 2009->2019: +237.5% (from 4710 to 15898)


## Save the title-level table for reuse

In [10]:
titles.to_csv(os.path.join(SUMMARY_DIR, "titles_level_table.csv"), index=False)
print(f"Saved title-level table: {titles.shape}")

Saved title-level table: (350, 7)


## Summary

| Question | Finding |
|---|---|
| One-hit-wonders vs. repeaters | 72.9% one-hit-wonders (255), 27.1% repeat (95) — top repeaters span **10 years** (tie: *The 5 Love Languages*, *APA Publication Manual*) |
| Genre skew | Negligible — Fiction repeats 27.5% of the time, Non Fiction 26.8% |
| Price vs. repeat status | Essentially no relationship (r=0.01); repeaters are if anything slightly *cheaper* ($12.28 vs $13.28 mean) |
| Rating vs. repeat status | Essentially no relationship (r=0.05); repeaters and one-hit-wonders rate almost identically (~4.6) |
| Reviews vs. repeat status | **The clearest signal** (r=0.23): repeaters average 16,381 reviews vs. 7,321 for one-hit-wonders — more than double |
| Top authors | Jeff Kinney leads on total appearances (12, all distinct titles — a franchise); Gary Chapman is the purest "single title, many years" repeater (11 appearances, only 2 titles) |
| 2009→2019 trend | Avg price fell 35% ($15.40→$10.08); avg rating rose slightly (+0.16); avg reviews more than tripled (+237%) |

Full write-up: [`docs/04_analyze.md`](../docs/04_analyze.md).